#YOLOv8 Setup

In [ ]:
"""
Second version.
Transfer learning actually working now.
Do not change this code, it'll be used as a case study for the next version.
"""

import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn import tasks
from collections import OrderedDict

# ===================================================================
# PART 1: DEFINE THE CUSTOM MODULES
# ===================================================================

# This module is unchanged
class ScaleAwareFeatureAggregation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.mult_scale_heads = 8
        self.query_conv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=7, stride=4)
        self.query_conv2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)
        self.key_conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)

    def forward(self, x, quarter_scale_x):
        orig_x = x
        x = self.query_conv1(x)
        x = self.query_conv2(x)
        quarter_scale_x = self.key_conv(quarter_scale_x)
        batch_size, C, roi_h, roi_w = x.size()
        x = x.view(batch_size, 1, C, roi_h, roi_w)
        quarter_scale_x = quarter_scale_x.view(batch_size, 1, C, roi_h, roi_w)
        x = torch.cat((x, quarter_scale_x), dim=1)
        batch_size, img_n, _, roi_h, roi_w = x.size()
        x_embed = x
        c_embed = x_embed.size(2)
        x_embed = x_embed.view(batch_size, img_n, self.mult_scale_heads, -1, roi_h, roi_w)
        target_x_embed = x_embed[:, [1]]
        ada_weights = torch.sum(x_embed * target_x_embed, dim=3, keepdim=True) / (float(c_embed / self.mult_scale_heads)**0.5)
        ada_weights = ada_weights.expand(-1, -1, -1, int(c_embed / self.mult_scale_heads), -1, -1).contiguous()
        ada_weights = ada_weights.view(batch_size, img_n, c_embed, roi_h, roi_w)
        ada_weights = ada_weights.softmax(dim=1)
        x = (x * ada_weights).sum(dim=1)
        aggregated_feature = F.interpolate(x, size=orig_x.shape[-2:], mode='bilinear', align_corners=False)
        aggregated_enhanced_representation = orig_x + aggregated_feature
        return aggregated_enhanced_representation

class FeatEnHancer(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatEnHancer, self).__init__()
        int_out_channels = 32
        out_channels = 24  # Used for intermediate features
        self.relu = nn.ReLU(inplace=True)

        self.e_conv1 = nn.Conv2d(in_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv2 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv3 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv4 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv5 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv6 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv7 = nn.Conv2d(int_out_channels * 2, out_channels, 3, 1, 1, bias=True)
        self.ue_conv8 = nn.Conv2d(out_channels * 2, int_out_channels, 3, 1, 1, bias=True)

        self.ue_conv9 = nn.Conv2d(int_out_channels, 3, 3, 1, 1, bias=True) #

        self.quarter_conv = nn.Conv2d(in_channels, in_channels, 7, 4)
        self.hexa_conv = nn.Conv2d(in_channels, in_channels, 3, 2)
        self.scale_aware_aggregation = ScaleAwareFeatureAggregation(channels=out_channels)

    def forward(self, x):
        original_x = x # Keep a reference to the original input

        # Forward pass through the enhancement network
        quarter_scale_x = self.quarter_conv(x); hexa_scale_x = self.hexa_conv(quarter_scale_x)
        x1 = self.relu(self.e_conv1(x)); quarter_scale_x1 = self.relu(self.e_conv1(quarter_scale_x)); hexa_scale_x1 = self.relu(self.e_conv1(hexa_scale_x))
        x2 = self.relu(self.e_conv2(x1)); quarter_scale_x2 = self.relu(self.e_conv2(quarter_scale_x1)); hexa_scale_x2 = self.relu(self.e_conv2(hexa_scale_x1))
        x3 = self.relu(self.e_conv3(x2)); quarter_scale_x3 = self.relu(self.e_conv3(quarter_scale_x2)); hexa_scale_x3 = self.relu(self.e_conv3(hexa_scale_x2))
        x4 = self.relu(self.e_conv4(x3)); quarter_scale_x4 = self.relu(self.e_conv4(quarter_scale_x3)); hexa_scale_x4 = self.relu(self.e_conv4(hexa_scale_x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1))); quarter_scale_x5 = self.relu(self.e_conv5(torch.cat([quarter_scale_x3, quarter_scale_x4], 1))); hexa_scale_x5 = self.relu(self.e_conv5(torch.cat([hexa_scale_x3, hexa_scale_x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1))); quarter_scale_x6 = self.relu(self.e_conv6(torch.cat([quarter_scale_x2, quarter_scale_x5], 1))); hexa_scale_x6 = self.relu(self.e_conv6(torch.cat([hexa_scale_x2, hexa_scale_x5], 1)))
        x7 = self.relu(self.e_conv7(torch.cat([x1, x6], 1))); quarter_scale_x7 = self.relu(self.e_conv7(torch.cat([quarter_scale_x1, quarter_scale_x6], 1))); hexa_scale_x7 = self.e_conv7(torch.cat([hexa_scale_x1, hexa_scale_x6], 1))
        x7 = self.scale_aware_aggregation(x7, quarter_scale_x7)
        hexa_scale_x7 = F.interpolate(hexa_scale_x7, size=x7.shape[-2:], mode='bilinear', align_corners=False)
        x8 = self.ue_conv8(torch.cat([x7, hexa_scale_x7], 1))

        enhancement_residual = torch.tanh(self.ue_conv9(x8))#

        # We return the original image with the enhancement added.
        return original_x + enhancement_residual

print("Custom module classes defined and modified.")

# ===================================================================
# PART 2: INJECT THE CUSTOM MODULE
# ===================================================================
setattr(tasks, 'FeatEnHancer', FeatEnHancer)
print("FeatEnHancer module successfully injected into Ultralytics.")

# ===================================================================
# PART 3: CREATE THE NEW MODEL YAML
# ===================================================================
# This YAML places FeatEnHancer at the beginning, then defines the
# rest of the yolov8m model exactly as it is in the standard configuration.
# This ensures layers 1...N in our model match layers 0...N-1 in the OG model.
yaml_content = """
# YOLOv8m model with custom FeatEnHancer input layer
# Parameters
nc: 1  # number of classes
depth_multiple: 0.67  # model depth multiple
width_multiple: 0.75  # layer channel multiple

# Backbone
backbone:
  # [from, number, module, args]
  - [-1, 1, FeatEnHancer, [3]]       # Layer 0: Custom input enhancement. Takes 3 channels, outputs 3.

  # The rest of this is the STANDARD yolov8m backbone.
  # These layers will now load the pretrained weights correctly.
  - [-1, 1, Conv, [64, 3, 2]]      # Layer 1 (was 0 in OG model)
  - [-1, 2, C2f, [128, True]]      # Layer 2 (was 1 in OG model)
  - [-1, 1, Conv, [128, 3, 2]]     # Layer 3 (was 2 in OG model)
  - [-1, 4, C2f, [256, True]]      # Layer 4 (was 3 in OG model)
  - [-1, 1, Conv, [256, 3, 2]]     # Layer 5 (was 4 in OG model)
  - [-1, 4, C2f, [512, True]]      # Layer 6 (was 5 in OG model)
  - [-1, 1, Conv, [512, 3, 2]]     # Layer 7 (was 6 in OG model)
  - [-1, 2, C2f, [512, True]]      # Layer 8 (was 7 in OG model)
  - [-1, 1, SPPF, [512, 5]]        # Layer 9 (was 8 in OG model)

# Head
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # Layer 10
  - [[-1, 6], 1, Concat, [1]]  # Concat with layer 6 (C2f)
  - [-1, 2, C2f, [512]]        # Layer 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # Layer 13
  - [[-1, 4], 1, Concat, [1]]  # Concat with layer 4 (C2f)
  - [-1, 2, C2f, [256]]        # Layer 15

  - [-1, 1, Conv, [256, 3, 2]] # Layer 16
  - [[-1, 12], 1, Concat, [1]] # Concat with layer 12
  - [-1, 2, C2f, [512]]        # Layer 18

  - [-1, 1, Conv, [512, 3, 2]] # Layer 19
  - [[-1, 9], 1, Concat, [1]]  # Concat with layer 9 (SPPF)
  - [-1, 2, C2f, [512]]        # Layer 21

  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""
model_yaml_filename = "Nyolov8m_featenhancer_transfer.yaml"
with open(model_yaml_filename, "w") as f:
    f.write(yaml_content)
print(f"'{model_yaml_filename}' created successfully.")

# ===================================================================
# PART 4: CUSTOM WEIGHT LOADING LOGIC
# ===================================================================
print("\n--- Starting Custom Weight Loading ---")

# Define paths
og_model_path = "/home/pdi/Downloads/Enhancement_detection/YOLO8/Results_131_ep/yolov8m_deepfish_ozfish_tensorboard/weights/best.pt"

# 1. Load the OG model with its weights
print(f"Loading OG model from: {og_model_path}")
og_model = YOLO(og_model_path)
og_state_dict = og_model.state_dict()
print("OG model weights loaded.")

# 2. Initialize the new custom model (it's empty)
print(f"Initializing new custom model from: {model_yaml_filename}")
custom_model = YOLO(model_yaml_filename)
custom_state_dict = custom_model.state_dict()
print("New custom model initialized.")

# 3. Create a new state dictionary to map the weights
new_state_dict = OrderedDict()
print("Mapping weights from OG model to new custom model...")

for k, v in og_state_dict.items():
    # Split the key name (e.g., 'model.0.conv.weight')
    key_parts = k.split('.')

    # Check if the key belongs to the main model sequence
    if key_parts[0] == 'model' and key_parts[1].isdigit():
        # Increment the layer index by 1 to account for the new FeatEnHancer at layer 0
        original_layer_idx = int(key_parts[1])
        new_layer_idx = original_layer_idx + 1
        key_parts[1] = str(new_layer_idx)

        # Reconstruct the new key
        new_key = ".".join(key_parts)

        # Check if this new key exists in our custom model and has the same shape
        if new_key in custom_state_dict and custom_state_dict[new_key].shape == v.shape:
            new_state_dict[new_key] = v
            # print(f"  Mapped {k} -> {new_key}") # Uncomment for verbose debugging
        else:
            print(f"  [!] Skipped {k}: Mismatch or key '{new_key}' not found in new model.")
    else:
        # For non-indexed parts of the model (like the final Detect layer), keep the key as is
        if k in custom_state_dict and custom_state_dict[k].shape == v.shape:
            new_state_dict[k] = v
            # print(f"  Mapped {k} -> {k} (no index change)") # Uncomment for verbose debugging

# 4. Load our carefully mapped weights into the custom model
custom_model.load_state_dict(new_state_dict, strict=False) # strict=False allows missing layers (i.e., our FeatEnHancer)
print("Weight transfer complete. FeatEnHancer layer remains randomly initialized.")

# ===================================================================
# PART 5: START TRAINING
# ===================================================================
data_yaml_path = "/home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/organized_dataset/dataset.yaml"
project_name = "Personal_Fish_Detection3"
run_name = "N2yolov8m_FeatEnHancer_with_Transfer"

print(f"\nStarting training using data from {data_yaml_path}...")

results = custom_model.train(
    data=data_yaml_path,
    epochs=150,
    imgsz=608,
    batch=8,
    optimizer='AdamW',
    lr0=0.0001,
    project=project_name,
    name=run_name,
    exist_ok=True,
    patience=50
)

print("Training finished.")

In [ ]:
"""
Second version.
Transfer learning actually working now.
Do not change this code, it'll be used as a case study for the next version.
"""

import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn import tasks
from collections import OrderedDict

# ===================================================================
# PART 1: DEFINE THE CUSTOM MODULES
# ===================================================================

# This module is unchanged
class ScaleAwareFeatureAggregation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.mult_scale_heads = 8
        self.query_conv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=7, stride=4)
        self.query_conv2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)
        self.key_conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)

    def forward(self, x, quarter_scale_x):
        orig_x = x
        x = self.query_conv1(x)
        x = self.query_conv2(x)
        quarter_scale_x = self.key_conv(quarter_scale_x)
        batch_size, C, roi_h, roi_w = x.size()
        x = x.view(batch_size, 1, C, roi_h, roi_w)
        quarter_scale_x = quarter_scale_x.view(batch_size, 1, C, roi_h, roi_w)
        x = torch.cat((x, quarter_scale_x), dim=1)
        batch_size, img_n, _, roi_h, roi_w = x.size()
        x_embed = x
        c_embed = x_embed.size(2)
        x_embed = x_embed.view(batch_size, img_n, self.mult_scale_heads, -1, roi_h, roi_w)
        target_x_embed = x_embed[:, [1]]
        ada_weights = torch.sum(x_embed * target_x_embed, dim=3, keepdim=True) / (float(c_embed / self.mult_scale_heads)**0.5)
        ada_weights = ada_weights.expand(-1, -1, -1, int(c_embed / self.mult_scale_heads), -1, -1).contiguous()
        ada_weights = ada_weights.view(batch_size, img_n, c_embed, roi_h, roi_w)
        ada_weights = ada_weights.softmax(dim=1)
        x = (x * ada_weights).sum(dim=1)
        aggregated_feature = F.interpolate(x, size=orig_x.shape[-2:], mode='bilinear', align_corners=False)
        aggregated_enhanced_representation = orig_x + aggregated_feature
        return aggregated_enhanced_representation

class FeatEnHancer(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatEnHancer, self).__init__()
        int_out_channels = 32
        out_channels = 24  # Used for intermediate features
        self.relu = nn.ReLU(inplace=True)

        self.e_conv1 = nn.Conv2d(in_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv2 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv3 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv4 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv5 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv6 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv7 = nn.Conv2d(int_out_channels * 2, out_channels, 3, 1, 1, bias=True)
        self.ue_conv8 = nn.Conv2d(out_channels * 2, int_out_channels, 3, 1, 1, bias=True)

        self.ue_conv9 = nn.Conv2d(int_out_channels, 3, 3, 1, 1, bias=True) #

        self.quarter_conv = nn.Conv2d(in_channels, in_channels, 7, 4)
        self.hexa_conv = nn.Conv2d(in_channels, in_channels, 3, 2)
        self.scale_aware_aggregation = ScaleAwareFeatureAggregation(channels=out_channels)

    def forward(self, x):
        original_x = x # Keep a reference to the original input

        # Forward pass through the enhancement network
        quarter_scale_x = self.quarter_conv(x); hexa_scale_x = self.hexa_conv(quarter_scale_x)
        x1 = self.relu(self.e_conv1(x)); quarter_scale_x1 = self.relu(self.e_conv1(quarter_scale_x)); hexa_scale_x1 = self.relu(self.e_conv1(hexa_scale_x))
        x2 = self.relu(self.e_conv2(x1)); quarter_scale_x2 = self.relu(self.e_conv2(quarter_scale_x1)); hexa_scale_x2 = self.relu(self.e_conv2(hexa_scale_x1))
        x3 = self.relu(self.e_conv3(x2)); quarter_scale_x3 = self.relu(self.e_conv3(quarter_scale_x2)); hexa_scale_x3 = self.relu(self.e_conv3(hexa_scale_x2))
        x4 = self.relu(self.e_conv4(x3)); quarter_scale_x4 = self.relu(self.e_conv4(quarter_scale_x3)); hexa_scale_x4 = self.relu(self.e_conv4(hexa_scale_x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1))); quarter_scale_x5 = self.relu(self.e_conv5(torch.cat([quarter_scale_x3, quarter_scale_x4], 1))); hexa_scale_x5 = self.relu(self.e_conv5(torch.cat([hexa_scale_x3, hexa_scale_x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1))); quarter_scale_x6 = self.relu(self.e_conv6(torch.cat([quarter_scale_x2, quarter_scale_x5], 1))); hexa_scale_x6 = self.relu(self.e_conv6(torch.cat([hexa_scale_x2, hexa_scale_x5], 1)))
        x7 = self.relu(self.e_conv7(torch.cat([x1, x6], 1))); quarter_scale_x7 = self.relu(self.e_conv7(torch.cat([quarter_scale_x1, quarter_scale_x6], 1))); hexa_scale_x7 = self.e_conv7(torch.cat([hexa_scale_x1, hexa_scale_x6], 1))
        x7 = self.scale_aware_aggregation(x7, quarter_scale_x7)
        hexa_scale_x7 = F.interpolate(hexa_scale_x7, size=x7.shape[-2:], mode='bilinear', align_corners=False)
        x8 = self.ue_conv8(torch.cat([x7, hexa_scale_x7], 1))

        enhancement_residual = torch.tanh(self.ue_conv9(x8))#

        # We return the original image with the enhancement added.
        return original_x + enhancement_residual

print("Custom module classes defined and modified.")

# ===================================================================
# PART 2: INJECT THE CUSTOM MODULE
# ===================================================================
setattr(tasks, 'FeatEnHancer', FeatEnHancer)
print("FeatEnHancer module successfully injected into Ultralytics.")

# ===================================================================
# PART 3: CREATE THE NEW MODEL YAML
# ===================================================================
# This YAML places FeatEnHancer at the beginning, then defines the
# rest of the yolov8m model exactly as it is in the standard configuration.
# This ensures layers 1...N in our model match layers 0...N-1 in the OG model.
yaml_content = """
# YOLOv8m model with custom FeatEnHancer input layer
# Parameters
nc: 1  # number of classes
depth_multiple: 0.67  # model depth multiple
width_multiple: 0.75  # layer channel multiple

# Backbone
backbone:
  # [from, number, module, args]
  - [-1, 1, FeatEnHancer, [3]]       # Layer 0: Custom input enhancement. Takes 3 channels, outputs 3.

  # The rest of this is the STANDARD yolov8m backbone.
  # These layers will now load the pretrained weights correctly.
  - [-1, 1, Conv, [64, 3, 2]]      # Layer 1 (was 0 in OG model)
  - [-1, 2, C2f, [128, True]]      # Layer 2 (was 1 in OG model)
  - [-1, 1, Conv, [128, 3, 2]]     # Layer 3 (was 2 in OG model)
  - [-1, 4, C2f, [256, True]]      # Layer 4 (was 3 in OG model)
  - [-1, 1, Conv, [256, 3, 2]]     # Layer 5 (was 4 in OG model)
  - [-1, 4, C2f, [512, True]]      # Layer 6 (was 5 in OG model)
  - [-1, 1, Conv, [512, 3, 2]]     # Layer 7 (was 6 in OG model)
  - [-1, 2, C2f, [512, True]]      # Layer 8 (was 7 in OG model)
  - [-1, 1, SPPF, [512, 5]]        # Layer 9 (was 8 in OG model)

# Head
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # Layer 10
  - [[-1, 6], 1, Concat, [1]]  # Concat with layer 6 (C2f)
  - [-1, 2, C2f, [512]]        # Layer 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # Layer 13
  - [[-1, 4], 1, Concat, [1]]  # Concat with layer 4 (C2f)
  - [-1, 2, C2f, [256]]        # Layer 15

  - [-1, 1, Conv, [256, 3, 2]] # Layer 16
  - [[-1, 12], 1, Concat, [1]] # Concat with layer 12
  - [-1, 2, C2f, [512]]        # Layer 18

  - [-1, 1, Conv, [512, 3, 2]] # Layer 19
  - [[-1, 9], 1, Concat, [1]]  # Concat with layer 9 (SPPF)
  - [-1, 2, C2f, [512]]        # Layer 21

  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""
model_yaml_filename = "Nyolov8m_featenhancer_transfer.yaml"
with open(model_yaml_filename, "w") as f:
    f.write(yaml_content)
print(f"'{model_yaml_filename}' created successfully.")

# ===================================================================
# PART 4: CUSTOM WEIGHT LOADING LOGIC
# ===================================================================
print("\n--- Starting Custom Weight Loading ---")

# Define paths
og_model_path = "/home/pdi/Downloads/Enhancement_detection/YOLO8/Results_131_ep/yolov8m_deepfish_ozfish_tensorboard/weights/best.pt"

# 1. Load the OG model with its weights
print(f"Loading OG model from: {og_model_path}")
og_model = YOLO(og_model_path)
og_state_dict = og_model.state_dict()
print("OG model weights loaded.")

# 2. Initialize the new custom model (it's empty)
print(f"Initializing new custom model from: {model_yaml_filename}")
custom_model = YOLO(model_yaml_filename)
custom_state_dict = custom_model.state_dict()
print("New custom model initialized.")

# 3. Create a new state dictionary to map the weights
new_state_dict = OrderedDict()
print("Mapping weights from OG model to new custom model...")

for k, v in og_state_dict.items():
    # Split the key name (e.g., 'model.0.conv.weight')
    key_parts = k.split('.')

    # Check if the key belongs to the main model sequence
    if key_parts[0] == 'model' and key_parts[1].isdigit():
        # Increment the layer index by 1 to account for the new FeatEnHancer at layer 0
        original_layer_idx = int(key_parts[1])
        new_layer_idx = original_layer_idx + 1
        key_parts[1] = str(new_layer_idx)

        # Reconstruct the new key
        new_key = ".".join(key_parts)

        # Check if this new key exists in our custom model and has the same shape
        if new_key in custom_state_dict and custom_state_dict[new_key].shape == v.shape:
            new_state_dict[new_key] = v
            # print(f"  Mapped {k} -> {new_key}") # Uncomment for verbose debugging
        else:
            print(f"  [!] Skipped {k}: Mismatch or key '{new_key}' not found in new model.")
    else:
        # For non-indexed parts of the model (like the final Detect layer), keep the key as is
        if k in custom_state_dict and custom_state_dict[k].shape == v.shape:
            new_state_dict[k] = v
            # print(f"  Mapped {k} -> {k} (no index change)") # Uncomment for verbose debugging

# 4. Load our carefully mapped weights into the custom model
custom_model.load_state_dict(new_state_dict, strict=False) # strict=False allows missing layers (i.e., our FeatEnHancer)
print("Weight transfer complete. FeatEnHancer layer remains randomly initialized.")

# ===================================================================
# PART 5: START TRAINING
# ===================================================================
data_yaml_path = "/home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/dataset.yaml"
project_name = "Personal_Fish_Detection2"
run_name = "Nyolov8m_FeatEnHancer_with_Transfer"

print(f"\nStarting training using data from {data_yaml_path}...")

results = custom_model.train(
    data=data_yaml_path,
    epochs=150,
    imgsz=608,
    batch=8,
    optimizer='AdamW',
    lr0=0.0001,
    project=project_name,
    name=run_name,
    exist_ok=True,
    patience=50
)

print("Training finished.")

Custom module classes defined and modified.
FeatEnHancer module successfully injected into Ultralytics.
'Nyolov8m_featenhancer_transfer.yaml' created successfully.

--- Starting Custom Weight Loading ---
Loading OG model from: /home/pdi/Downloads/Enhancement_detection/YOLO8/Results_131_ep/yolov8m_deepfish_ozfish_tensorboard/weights/best.pt
OG model weights loaded.
Initializing new custom model from: Nyolov8m_featenhancer_transfer.yaml
New custom model initialized.
Mapping weights from OG model to new custom model...
Weight transfer complete. FeatEnHancer layer remains randomly initialized.

Starting training using data from /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/dataset.yaml...
New https://pypi.org/project/ultralytics/8.3.153 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.152 🚀 Python-3.10.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11900MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto

100%|██████████| 5.35M/5.35M [00:00<00:00, 34.5MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5521.5±4427.9 MB/s, size: 287.4 KB)


train: Scanning /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/train/labels.cache... 3716 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3716/3716 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1377.6±269.4 MB/s, size: 329.1 KB)


...

    135/150        11G      1.148     0.9263      1.221         14        608: 100%|██████████| 465/465 [04:14<00:00,  1.83it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [00:13<00:00,  4.31it/s]

                   all        930       1901      0.854      0.555      0.609      0.352
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 85, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



135 epochs completed in 10.197 hours.
Optimizer stripped from Personal_Fish_Detection2/Nyolov8m_FeatEnHancer_with_Transfer/weights/last.pt, 30.7MB
Optimizer stripped from Personal_Fish_Detection2/Nyolov8m_FeatEnHancer_with_Transfer/weights/best.pt, 30.7MB

Validating Personal_Fish_Detection2/Nyolov8m_FeatEnHancer_with_Transfer/weights/best.pt...
Ultralytics 8.3.152 🚀 Python-3.10.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11900MiB)
NYOLOv8m_featenhancer_transfer summary (fused): 90 layers, 15,176,934 parameters, 0 gradients, 294.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [00:13<00:00,  4.46it/s]


                   all        930       1901      0.856      0.548      0.617      0.359
Speed: 0.1ms preprocess, 12.7ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to Personal_Fish_Detection2/Nyolov8m_FeatEnHancer_with_Transfer
Training finished.


In [ ]:
"""
First version, did not work properly due to incorrecly loading
pre-trained weights. Do not run it, keeping it as a study case.
"""
# ===================================================================
# FINAL SIMPLIFIED SCRIPT (Using correct dataset.yaml path)
# ===================================================================
import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn import tasks

# --- PART 1: DEFINE THE CUSTOM MODULES (Same as before) ---
class ScaleAwareFeatureAggregation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.mult_scale_heads = 8
        self.query_conv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=7, stride=4)
        self.query_conv2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)
        self.key_conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=2)

    def forward(self, x, quarter_scale_x):
        orig_x = x; x = self.query_conv1(x); x = self.query_conv2(x); quarter_scale_x = self.key_conv(quarter_scale_x)
        batch_size, C, roi_h, roi_w = x.size()
        x = x.view(batch_size, 1, C, roi_h, roi_w); quarter_scale_x = quarter_scale_x.view(batch_size, 1, C, roi_h, roi_w)
        x = torch.cat((x, quarter_scale_x), dim=1)
        batch_size, img_n, _, roi_h, roi_w = x.size(); x_embed = x; c_embed = x_embed.size(2)
        x_embed = x_embed.view(batch_size, img_n, self.mult_scale_heads, -1, roi_h, roi_w)
        target_x_embed = x_embed[:, [1]]
        ada_weights = torch.sum(x_embed * target_x_embed, dim=3, keepdim=True) / (float(c_embed / self.mult_scale_heads)**0.5)
        ada_weights = ada_weights.expand(-1, -1, -1, int(c_embed / self.mult_scale_heads), -1, -1).contiguous()
        ada_weights = ada_weights.view(batch_size, img_n, c_embed, roi_h, roi_w)
        ada_weights = ada_weights.softmax(dim=1)
        x = (x * ada_weights).sum(dim=1)
        aggregated_feature = F.interpolate(x, size=orig_x.shape[-2:], mode='bilinear', align_corners=False)
        aggregated_enhanced_representation = orig_x + aggregated_feature
        return aggregated_enhanced_representation

class FeatEnHancer(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatEnHancer, self).__init__()
        int_out_channels = 32; out_channels = 24; self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(in_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv2 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv3 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv4 = nn.Conv2d(int_out_channels, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv5 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv6 = nn.Conv2d(int_out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.e_conv7 = nn.Conv2d(int_out_channels * 2, out_channels, 3, 1, 1, bias=True)
        self.ue_conv8 = nn.Conv2d(out_channels * 2, int_out_channels, 3, 1, 1, bias=True)
        self.ue_conv9 = nn.Conv2d(int_out_channels, out_channels, 3, 1, 1, bias=True)
        self.quarter_conv = nn.Conv2d(in_channels, in_channels, 7, 4)
        self.hexa_conv = nn.Conv2d(in_channels, in_channels, 3, 2)
        self.scale_aware_aggregation = ScaleAwareFeatureAggregation(channels=out_channels)
    def forward(self, x):
        quarter_scale_x = self.quarter_conv(x); hexa_scale_x = self.hexa_conv(quarter_scale_x)
        x1 = self.relu(self.e_conv1(x)); quarter_scale_x1 = self.relu(self.e_conv1(quarter_scale_x)); hexa_scale_x1 = self.relu(self.e_conv1(hexa_scale_x))
        x2 = self.relu(self.e_conv2(x1)); quarter_scale_x2 = self.relu(self.e_conv2(quarter_scale_x1)); hexa_scale_x2 = self.relu(self.e_conv2(hexa_scale_x1))
        x3 = self.relu(self.e_conv3(x2)); quarter_scale_x3 = self.relu(self.e_conv3(quarter_scale_x2)); hexa_scale_x3 = self.relu(self.e_conv3(hexa_scale_x2))
        x4 = self.relu(self.e_conv4(x3)); quarter_scale_x4 = self.relu(self.e_conv4(quarter_scale_x3)); hexa_scale_x4 = self.relu(self.e_conv4(hexa_scale_x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1))); quarter_scale_x5 = self.relu(self.e_conv5(torch.cat([quarter_scale_x3, quarter_scale_x4], 1))); hexa_scale_x5 = self.relu(self.e_conv5(torch.cat([hexa_scale_x3, hexa_scale_x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1))); quarter_scale_x6 = self.relu(self.e_conv6(torch.cat([quarter_scale_x2, quarter_scale_x5], 1))); hexa_scale_x6 = self.relu(self.e_conv6(torch.cat([hexa_scale_x2, hexa_scale_x5], 1)))
        x7 = self.relu(self.e_conv7(torch.cat([x1, x6], 1))); quarter_scale_x7 = self.relu(self.e_conv7(torch.cat([quarter_scale_x1, quarter_scale_x6], 1))); hexa_scale_x7 = self.e_conv7(torch.cat([hexa_scale_x1, hexa_scale_x6], 1))
        x7 = self.scale_aware_aggregation(x7, quarter_scale_x7)
        hexa_scale_x7 = F.interpolate(hexa_scale_x7, size=x7.shape[-2:], mode='bilinear', align_corners=False)
        x8 = self.ue_conv8(torch.cat([x7, hexa_scale_x7], 1))
        activated_enhanced_feature = torch.tanh(self.ue_conv9(x8))
        feature1, feature2, feature3, feature4, feature5, feature6, feature7, feature8 = torch.split(activated_enhanced_feature, 3, dim=1)
        x = x + feature1; x = x + feature2; x = x + feature3; x = x + feature4;
        x = x + feature5; x = x + feature6; x = x + feature7;
        enhanced_representation = x + feature8
        return enhanced_representation
print("Custom module classes defined.")

# --- PART 2: INJECT THE CUSTOM MODULE ---
setattr(tasks, 'FeatEnHancer', FeatEnHancer)
print("FeatEnHancer module successfully injected.")

# --- PART 3: CREATE THE MODEL YAML ---
yaml_content = """
nc: 1
depth_multiple: 0.67
width_multiple: 0.75
backbone:
  - [-1, 1, FeatEnHancer, []]
  - [-1, 1, Conv, [48, 3, 2]]
  - [-1, 1, Conv, [96, 3, 2]]
  - [-1, 2, C2f, [96, True]]
  - [-1, 1, Conv, [192, 3, 2]]
  - [-1, 4, C2f, [192, True]]
  - [-1, 1, Conv, [384, 3, 2]]
  - [-1, 4, C2f, [384, True]]
  - [-1, 1, Conv, [576, 3, 2]]
  - [-1, 2, C2f, [576, True]]
  - [-1, 1, SPPF, [576, 5]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 2, C2f, [576]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 2, C2f, [384]]
  - [-1, 1, Conv, [384, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C2f, [576]]
  - [-1, 1, Conv, [576, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C2f, [576]]
  - [[16, 19, 22], 1, Detect, [nc]]
"""
model_yaml_filename = "yolov8m_full_transfer.yaml"
with open(model_yaml_filename, "w") as f:
    f.write(yaml_content)
print(f"'{model_yaml_filename}' created successfully.")

# --- PART 4: INITIALIZE MODEL AND LOAD WEIGHTS ---
model = YOLO(model_yaml_filename)
pretrained_weights_path = "/home/pdi/Downloads/Enhancement_detection/YOLO8/Results_131_ep/yolov8m_deepfish_ozfish_tensorboard/weights/best.pt"
model.load(pretrained_weights_path)
print("Model created and pre-trained weights loaded.")

# --- PART 5: DEFINE DATASET PATH AND START TRAINING ---

# CORRECTED: The path now points to the dataset.yaml file inside the 'TrainVal' directory
data_yaml_path = "/home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/dataset.yaml"

project_name = "Personal_Fish_Detection"
run_name = "yolov8m_FeatEnHancer_Final_Train"

print(f"\\nStarting training using data from {data_yaml_path}...")

results = model.train(
    data=data_yaml_path,  # <-- Use the direct path to your dataset's yaml
    epochs=150,
    imgsz=608,
    batch=8,
    optimizer='AdamW',
    lr0=0.0001,
    project=project_name,
    name=run_name,
    exist_ok=True,
    patience=50
)

print("Training finished.")

Custom module classes defined.
FeatEnHancer module successfully injected.
'yolov8m_full_transfer.yaml' created successfully.
Transferred 3/407 items from pretrained weights
Model created and pre-trained weights loaded.
\nStarting training using data from /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/dataset.yaml...
New https://pypi.org/project/ultralytics/8.3.153 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.152 🚀 Python-3.10.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11900MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=

train: Scanning /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/train/labels... 3716 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3716/3716 [00:00<00:00, 4821.79it/s]

train: New cache created: /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/train/labels.cache


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2245.9±1799.2 MB/s, size: 329.1 KB)


val: Scanning /home/pdi/Downloads/Enhancement_detection/Datasets/FishTrack23/New_Train_Data_Split/val/labels... 930 images, 0 backgrounds, 0 corrupt: 100%|██████████| 930/930 [00:00<00:00, 2593.86it/s]

...

    150/150      8.16G      1.243     0.9125      1.087          7        608: 100%|██████████| 465/465 [03:11<00:00,  2.43it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [00:10<00:00,  5.56it/s]

                   all        930       1901      0.826      0.512      0.569      0.313



150 epochs completed in 8.445 hours.
Optimizer stripped from Personal_Fish_Detection/yolov8m_FeatEnHancer_Final_Train/weights/last.pt, 38.1MB
Optimizer stripped from Personal_Fish_Detection/yolov8m_FeatEnHancer_Final_Train/weights/best.pt, 38.1MB

Validating Personal_Fish_Detection/yolov8m_FeatEnHancer_Final_Train/weights/best.pt...
Ultralytics 8.3.152 🚀 Python-3.10.13 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11900MiB)
YOLOv8m_full_transfer summary (fused): 91 layers, 18,908,467 parameters, 0 gradients, 157.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [00:10<00:00,  5.63it/s]


                   all        930       1901      0.828      0.536      0.583      0.322
Speed: 0.1ms preprocess, 9.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to Personal_Fish_Detection/yolov8m_FeatEnHancer_Final_Train
Training finished.
